In [1]:
import os
import requests

os.makedirs("data", exist_ok=True)

FILES = {
    "product.csv": "https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/product.csv",
    "label.csv":   "https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/label.csv",
    "queries.csv": "https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/query.csv",
}

for name, url in FILES.items():
    path = os.path.join("data", name)
    if os.path.exists(path):
        print(f"{name}: уже есть, пропускаем")
        continue
    print(f"Скачиваем {name}...")
    r = requests.get(url)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    print(f"  -> {path} ({os.path.getsize(path) / 1e6:.1f} MB)")

product.csv: уже есть, пропускаем
label.csv: уже есть, пропускаем
queries.csv: уже есть, пропускаем


In [2]:
with open("data/product.csv", "rb") as f:
    head = f.read(600)
print(repr(head))

b'product_id\tproduct_name\tproduct_class\tcategory hierarchy\tproduct_description\tproduct_features\trating_count\taverage_rating\treview_count\n0\tsolid wood platform bed\tBeds\tFurniture / Bedroom Furniture / Beds & Headboards / Beds / Twin Beds\tgood , deep sleep can be quite difficult to have in this busy age . fortunately , there \xe2\x80\x99 s an antidote to such a problem : a nice , quality bed frame like the acacia kaylin . solidly constructed from acacia wood , this bed frame will stand the test of time and is fit to rest your shoulders on for years and years . its sleek , natural wood grain appearance prov'


In [3]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)

products = pd.read_csv("data/product.csv", sep="\t")
queries  = pd.read_csv("data/queries.csv",  sep="\t")
labels   = pd.read_csv("data/label.csv",    sep="\t")

print("products:", products.shape)
print("queries: ", queries.shape)
print("labels:  ", labels.shape)
print()
print("Колонки product.csv:", list(products.columns))

products: (42994, 9)
queries:  (480, 3)
labels:   (233448, 4)

Колонки product.csv: ['product_id', 'product_name', 'product_class', 'category hierarchy', 'product_description', 'product_features', 'rating_count', 'average_rating', 'review_count']


In [4]:
products.info()

# Сколько пропусков в каждом поле
products.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42994 entries, 0 to 42993
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           42994 non-null  int64  
 1   product_name         42994 non-null  object 
 2   product_class        40142 non-null  object 
 3   category hierarchy   41438 non-null  object 
 4   product_description  36986 non-null  object 
 5   product_features     42994 non-null  object 
 6   rating_count         33542 non-null  float64
 7   average_rating       33542 non-null  float64
 8   review_count         33542 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 3.0+ MB


product_id                0
product_name              0
product_class          2852
category hierarchy     1556
product_description    6008
product_features          0
rating_count           9452
average_rating         9452
review_count           9452
dtype: int64

In [5]:
sample = products.sample(1, random_state=42).iloc[0]
for col in products.columns:
    val = sample[col]
    # Длинные описания обрежем, чтобы было читаемо
    if isinstance(val, str) and len(val) > 400:
        val = val[:400] + " ... [обрезано]"
    print(f"--- {col} ---\n{val}\n")

--- product_id ---
32193

--- product_name ---
woven wire basket

--- product_class ---
Boxes, Bins, Baskets, & Buckets

--- category hierarchy ---
Storage & Organization / Storage Containers & Drawers / All Storage Containers

--- product_description ---
nan

--- product_features ---
overalldepth-fronttoback:18|producttype : basket|features : stackable|overallheight-toptobottom:11.25|overalldepth-fronttoback:14|overallheight-toptobottom:16.75|style : coastal|lifestage : adult|overallwidth-sidetoside:10|primarymaterial : wicker|shape : round|overallwidth-sidetoside:18|overalldepth-fronttoback:10|overallheight-toptobottom:20.5|color : brown|overallproductweight:4|overallwidth-si ... [обрезано]

--- rating_count ---
21.0

--- average_rating ---
5.0

--- review_count ---
19.0



In [6]:
for i in range(3):
    print(f"--- товар {i} ---")
    print(repr(products["product_features"].iloc[i]))
    print()

--- товар 0 ---
'overallwidth-sidetoside:64.7|dsprimaryproductstyle : modern|countryoforigin-additionaldetails : made in viet nam|headboardtype : slat|headboardheight-toptobedframe:25.35|weightcapacity:500|estimatedtimetosetup:120|overalllength-headtotoe:77.9|levelofassembly : full assembly needed|overallproductweight:78.7|woodspecies : rubberwood|numberofslatsincluded:12|overallheight-toptobottom:39.6|headboardshape : rectangular|woodspecies : acacia|weightcapacity:800|baseconstruction : platform|color : caramel|additionaltoolsrequirednotincluded : all tools included|headboarddepth-fronttoback:0.78|overallwidth-sidetoside:43.3|color : chocolate|warrantylength:5 years|bedtype : other|mattresssize : queen|bedstorageincluded : none|purposefuldistressingtype : wood scoring & rub through|overalllength-headtotoe:83|recommendedmattressthickness:10|overallproductweight:52|mattresssize : twin|overallheightoflegs:14.25|clearancefromfloortoundersideofbed:14|dssecondaryproductstyle : mid-century 

In [7]:
# Доля товаров с пустыми/отсутствующими features
empty = products["product_features"].isna() | (products["product_features"] == "")
print(f"Товаров без features: {empty.sum()} из {len(products)}")

Товаров без features: 0 из 42994


In [8]:
queries.head(10)
labels.head(10)

# Сколько уникальных запросов размечено, и распределение классов релевантности
print("Уникальных запросов в queries:", queries["query_id"].nunique())
print("Уникальных запросов с разметкой:", labels["query_id"].nunique())
print()
print("Распределение меток релевантности:")
print(labels["label"].value_counts())

Уникальных запросов в queries: 480
Уникальных запросов с разметкой: 480

Распределение меток релевантности:
label
Partial       146633
Irrelevant     61201
Exact          25614
Name: count, dtype: int64


In [9]:
# Проверим, что разметка ссылается на реальные product_id
missing_ids = ~labels["product_id"].isin(products["product_id"])
print(f"Меток, ссылающихся на отсутствующие товары: {missing_ids.sum()}")

Меток, ссылающихся на отсутствующие товары: 0


In [10]:
print("Колонки queries.csv:", list(queries.columns))
queries.head(15)

Колонки queries.csv: ['query_id', 'query', 'query_class']


,query_id,query,query_class
0,0,salon chair,Massage Chairs
1,1,smart coffee table,Coffee & Cocktail Tables
2,2,dinosaur,Kids Wall Décor
3,3,turquoise pillows,Accent Pillows
4,4,chair and a half recliner,Recliners
5,5,sofa with ottoman,Sectionals
6,6,acrylic clear chair,Dining Chairs
7,7,driftwood mirror,Wall & Accent Mirrors
8,8,home sweet home sign,Wall Décor
9,9,coffee table fire pit,Outdoor Fireplaces


In [11]:
queries["query"].str.len().describe()

count    480.000000
mean      20.979167
std        9.211252
min        3.000000
25%       14.000000
50%       20.000000
75%       25.000000
max       67.000000
Name: query, dtype: float64

In [12]:
counts = labels.groupby(["query_id", "label"]).size().unstack(fill_value=0)
counts["total"] = counts.sum(axis=1)

print(f"Запросов без Exact: {(counts['Exact'] == 0).sum()} из {len(counts)}")
print(f"Запросов с <= 2 Exact: {(counts['Exact'] <= 2).sum()}")
print(f"Запросов с >= 10 Exact: {(counts['Exact'] >= 10).sum()}")
counts[["Exact", "Partial", "Irrelevant", "total"]].describe().round(1)

Запросов без Exact: 101 из 480
Запросов с <= 2 Exact: 169
Запросов с >= 10 Exact: 243


label,Exact,Partial,Irrelevant,total
count,480.0,480.0,480.0,480.0
mean,53.4,305.5,127.5,486.4
std,105.5,504.0,346.6,708.5
min,0.0,0.0,0.0,1.0
25%,1.0,67.8,3.0,122.8
50%,10.0,126.0,18.5,216.5
75%,63.0,294.5,73.2,523.8
max,878.0,3438.0,2940.0,4329.0


In [13]:
# Возьмём запрос с ~5 Exact — достаточно наглядно, но не перегруз
cand = counts[counts["Exact"].between(3, 8)].index
qid = cand[0]
qtext = queries.loc[queries["query_id"] == qid, "query"].iloc[0]
print(f"query_id={qid} | запрос: {qtext!r}\n")

merged = labels[labels["query_id"] == qid].merge(
    products[["product_id", "product_name", "product_class"]], on="product_id"
)
for lab in ["Exact", "Partial", "Irrelevant"]:
    sub = merged[merged["label"] == lab]
    print(f"--- {lab} ({len(sub)} шт.) ---")
    for _, r in sub.head(6).iterrows():
        print(f"  [{r['product_class']}] {r['product_name']}")
    print()

query_id=1 | запрос: 'smart coffee table'

--- Exact (8 шт.) ---
  [Coffee & Cocktail Tables] addaly abstract coffee table
  [Coffee & Cocktail Tables] adoni coffee table
  [Coffee & Cocktail Tables] celaya solid wood drum coffee table
  [Coffee & Cocktail Tables] clevedon sled coffee table
  [Coffee & Cocktail Tables] glastonbury coffee table
  [Coffee & Cocktail Tables] milivoje coffee table with storage

--- Partial (1410 шт.) ---
  [Coffee & Cocktail Tables] abheer floor shelf coffee table
  [Coffee & Cocktail Tables] ahart frame coffee table
  [Coffee & Cocktail Tables] ahern coffee table
  [Coffee & Cocktail Tables] akrati coffee table
  [Coffee & Cocktail Tables] alberte coffee table
  [Coffee & Cocktail Tables] aleeshia lift top trestle coffee table

--- Irrelevant (18 шт.) ---
  [Dining Bowls] allagash 8 oz . salad bowl
  [nan] aerial breakfast bed bamboo nesting 3 piece coffee table tray set
  [Decorative Trays] poulan 4 piece coffee table tray set
  [Decorative Trays] servin

In [14]:
def parse_features(raw: str):
    """'attr1:v1|attr2:v2|...' -> list of (attr, value). Терпит мусор."""
    if not isinstance(raw, str):
        return []
    out = []
    for chunk in raw.split("|"):
        if ":" not in chunk:
            continue
        attr, val = chunk.split(":", 1)
        attr, val = attr.strip().lower(), val.strip().lower()
        if attr and val:                       # отбрасываем ` : no`
            out.append((attr, val))
    return out

# Прогоняем по всему каталогу — если упадёт, увидим где
parsed = products["product_features"].map(parse_features)
print("Распарсено без ошибок:", parsed.map(len).sum(), "пар атрибут/значение")
print("Медиана пар на товар:", int(parsed.map(len).median()))
print("Максимум пар на товар:", parsed.map(len).max())

# Первые 10 пар у одного товара — глазами
sample_idx = parsed.map(len).idxmax()
print(f"\nТовар с максимумом атрибутов (id={products.loc[sample_idx,'product_id']}):")
for a, v in parsed.loc[sample_idx][:15]:
    print(f"  {a!r} -> {v!r}")

Распарсено без ошибок: 2379336 пар атрибут/значение
Медиана пар на товар: 54
Максимум пар на товар: 402

Товар с максимумом атрибутов (id=14506):
  'rugsize' -> "runner 4 ' x 22'"
  'rugsize' -> "rectangle 11 ' x 16'"
  'rugsize' -> "half circle 1 ' 8 '' x 3 ' 4 ''"
  'overallproductweight' -> '99'
  'rugsize' -> "runner 3 ' x 42'"
  'overallproductweight' -> '7'
  'rugsize' -> "octagon 9 ' x 9'"
  'overalllength-endtoend' -> '54'
  'rugsize' -> "runner 3 ' x 26'"
  'rugsize' -> "round 2'"
  'rugsize' -> "hexagon 4 ' x 4'"
  'overalllength-endtoend' -> '72'
  'rugsize' -> "runner 2 ' x 28'"
  'rugsize' -> "runner 2 ' x 20'"
  'rugsize' -> "hexagon 11 ' x 11'"


In [15]:
for col in ["product_name", "product_description", "category hierarchy"]:
    s = products[col].dropna().astype(str)
    print(f"{col}:")
    print(f"  заполнено: {len(s)}/{len(products)}")
    print(f"  длина (символы): min={s.str.len().min()}, median={int(s.str.len().median())}, max={s.str.len().max()}")
    print()

product_name:
  заполнено: 42994/42994
  длина (символы): min=3, median=37, max=257

product_description:
  заполнено: 36986/42994
  длина (символы): min=6, median=418, max=4060

category hierarchy:
  заполнено: 41438/42994
  длина (символы): min=5, median=68, max=209



In [16]:
import re
from nltk.stem import PorterStemmer

_stem = PorterStemmer()
# Токен: буквы/цифры, опционально с десятичной частью (14.2 — один токен)
_tok_re = re.compile(r"[a-z0-9]+(?:\.[0-9]+)?")

def tokenize(text):
    if not isinstance(text, str):
        return []
    # Числа не стеммим, остальное — Porter
    return [_stem.stem(t) if not t[0].isdigit() else t
            for t in _tok_re.findall(text.lower())]

# Проверка на пограничных случаях
for s in ["smart coffee table", "Queen-size Beds", "14.2 inch wide", "don't stop", "1 year"]:
    print(f"{s!r:30s} -> {tokenize(s)}")

'smart coffee table'           -> ['smart', 'coffe', 'tabl']
'Queen-size Beds'              -> ['queen', 'size', 'bed']
'14.2 inch wide'               -> ['14.2', 'inch', 'wide']
"don't stop"                   -> ['don', 't', 'stop']
'1 year'                       -> ['1', 'year']


In [17]:
def parse_features(raw):
    """'attr1:v1|attr2:v2' -> [(attr, val), ...]. Терпит пустые имена."""
    if not isinstance(raw, str):
        return []
    out = []
    for chunk in raw.split("|"):
        if ":" not in chunk:
            continue
        attr, val = chunk.split(":", 1)
        attr, val = attr.strip().lower(), val.strip().lower()
        if attr and val:
            out.append((attr, val))
    return out

In [18]:
from tqdm import tqdm

def build_doc(row, name_boost=2):
    parts = [str(row["product_name"])] * name_boost
    for k, v in parse_features(row["product_features"]):
        parts.append(f"{k} {v}")
    if isinstance(row["product_description"], str):
        parts.append(row["product_description"])
    if isinstance(row["product_class"], str):
        parts.append(row["product_class"])
    return " ".join(parts)

tqdm.pandas()
products["doc_bm25"] = products.progress_apply(build_doc, axis=1)
print("Median doc length (chars):", int(products["doc_bm25"].str.len().median()))
print("Max doc length (chars):   ", products["doc_bm25"].str.len().max())

100%|██████████| 42994/42994 [00:03<00:00, 11433.44it/s]

Median doc length (chars): 1897
Max doc length (chars):    11154


In [19]:
import bm25s
import time

product_ids = products["product_id"].tolist()

t0 = time.time()
corpus_tokens = [tokenize(d) for d in tqdm(products["doc_bm25"], desc="tokenize")]
print(f"Токенизация корпуса: {time.time() - t0:.1f}s")

t0 = time.time()
retriever = bm25s.BM25()
retriever.index(corpus_tokens)
print(f"Построение BM25-индекса: {time.time() - t0:.1f}s")

tokenize: 100%|██████████| 42994/42994 [02:38<00:00, 270.87it/s]

Токенизация корпуса: 158.7s


BM25S Create Vocab:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/42994 [00:00<?, ?it/s]

Построение BM25-индекса: 9.1s


In [20]:
import numpy as np
from collections import defaultdict

# qrels: {query_id: {product_id: gain}}, gain: Exact=2, Partial=1, Irrelevant=0
gain_map = {"Exact": 2, "Partial": 1, "Irrelevant": 0}
label_map = defaultdict(dict)
for _, r in labels.iterrows():
    label_map[r["query_id"]][r["product_id"]] = gain_map[r["label"]]

def _idcg(gains, k):
    ideal = sorted(gains.values(), reverse=True)[:k]
    return sum(g / np.log2(i + 2) for i, g in enumerate(ideal))

def ndcg_at_k_pessimistic(ranked_ids, qid, k=10):
    gains = label_map.get(qid, {})
    if not gains:
        return 0.0
    top = ranked_ids[:k]
    dcg = sum(gains.get(pid, 0) / np.log2(i + 2) for i, pid in enumerate(top))
    idcg = _idcg(gains, k)
    return dcg / idcg if idcg > 0 else 0.0

def ndcg_at_k_pooled(ranked_ids, qid, k=10):
    gains = label_map.get(qid, {})
    if not gains:
        return 0.0
    filtered = [pid for pid in ranked_ids if pid in gains][:k]
    if not filtered:
        return 0.0
    dcg = sum(gains[pid] / np.log2(i + 2) for i, pid in enumerate(filtered))
    idcg = _idcg(gains, k)
    return dcg / idcg if idcg > 0 else 0.0

In [21]:
import time
import numpy as np

query_texts = queries["query"].tolist()
query_ids   = queries["query_id"].tolist()

t0 = time.time()
query_tokens = [tokenize(q) for q in query_texts]
results_idx, scores = retriever.retrieve(query_tokens, k=100)
print(f"Retrieval: {time.time() - t0:.1f}s | shape={results_idx.shape}")

# Мапим индексы корпуса -> product_id, отбрасываем нулевые скоры
results = {}
for i, qid in enumerate(query_ids):
    results[qid] = [product_ids[j] for j, s in zip(results_idx[i], scores[i]) if s > 0]

ndcg_pess, ndcg_pool, judged_top10 = [], [], []
for qid in query_ids:
    ranked = results[qid]
    ndcg_pess.append(ndcg_at_k_pessimistic(ranked, qid))
    ndcg_pool.append(ndcg_at_k_pooled(ranked, qid))
    judged_top10.append(sum(1 for pid in ranked[:10] if pid in label_map.get(qid, {})))

print()
print("=== Baseline BM25: name×2 + features + description + class ===")
print(f"NDCG@10 (pessimistic): mean={np.mean(ndcg_pess):.4f}  median={np.median(ndcg_pess):.4f}")
print(f"NDCG@10 (pooled):      mean={np.mean(ndcg_pool):.4f}  median={np.median(ndcg_pool):.4f}")
print(f"Judged docs in top-10: mean={np.mean(judged_top10):.2f} / 10")

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

Retrieval: 0.7s | shape=(480, 100)

=== Baseline BM25: name×2 + features + description + class ===
NDCG@10 (pessimistic): mean=0.7127  median=0.7952
NDCG@10 (pooled):      mean=0.8150  median=0.9149
Judged docs in top-10: mean=8.37 / 10


In [22]:
pid_to_name = dict(zip(products["product_id"], products["product_name"]))

demo_qid = 1
demo_tokens = tokenize(queries.loc[queries["query_id"] == demo_qid, "query"].iloc[0])
demo_results, demo_scores = retriever.retrieve([demo_tokens], k=20)

print(f"Query: {queries.loc[queries['query_id'] == demo_qid, 'query'].iloc[0]!r}\n")
for j, (idx, sc) in enumerate(zip(demo_results[0], demo_scores[0]), 1):
    pid = product_ids[idx]
    lab = label_map[demo_qid].get(pid)
    lab_str = {2: "Exact", 1: "Partial", 0: "Irrelevant", None: "UNJUDGED"}[lab]
    print(f"  {j:2d}. [{lab_str:10s}] {pid_to_name[pid][:70]}  (score={sc:.2f})")

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'smart coffee table'

   1. [Exact     ] smart coffee table with storage  (score=6.13)
   2. [Partial   ] shaker coffee table  (score=5.67)
   3. [Partial   ] sanibel 3 legs coffee table  (score=5.57)
   4. [Partial   ] crotzer coffee table  (score=5.51)
   5. [Partial   ] sobro smart end table with built-in outlets  (score=5.40)
   6. [Partial   ] vandergriff wooden coffee table  (score=5.24)
   7. [UNJUDGED  ] stainless steel 3 piece coffee , tea , & sugar set  (score=4.60)
   8. [UNJUDGED  ] laguna 5 piece teak sofa seating group with cushions  (score=4.36)
   9. [UNJUDGED  ] bondi 4 piece sofa seating group set  (score=4.31)
  10. [UNJUDGED  ] litedge wi-fi accessible smart power strip socket plate  (score=4.17)
  11. [UNJUDGED  ] smart socket with usb port  (score=4.11)
  12. [UNJUDGED  ] smart propane scale for bbq gauge levels , bluetooth  (score=4.07)
  13. [UNJUDGED  ] alexa smart wifi plug with usb port  (score=4.05)
  14. [UNJUDGED  ] 4.5 cu . ft. smart front load was

In [23]:
import numpy as np

def evaluate_retrieval(ranked_dict, k_values=(1, 5, 10)):
    """ranked_dict: {qid: [pid, ...]}. Возвращает словарь метрик."""
    out = {}
    for k in k_values:
        pess = [ndcg_at_k_pessimistic(ranked_dict[qid], qid, k) for qid in query_ids]
        pool = [ndcg_at_k_pooled(ranked_dict[qid], qid, k) for qid in query_ids]
        out[f"NDCG@{k}_pess"] = {"mean": float(np.mean(pess)), "median": float(np.median(pess))}
        out[f"NDCG@{k}_pool"] = {"mean": float(np.mean(pool)), "median": float(np.median(pool))}

    # MRR: 1 / rank первого Exact в выдаче
    mrr = []
    for qid in query_ids:
        rr = 0.0
        for rank, pid in enumerate(ranked_dict[qid], 1):
            if label_map.get(qid, {}).get(pid) == 2:
                rr = 1.0 / rank
                break
        mrr.append(rr)
    out["MRR_exact"] = {"mean": float(np.mean(mrr)), "median": float(np.median(mrr))}
    return out

def print_metrics(m, title=""):
    if title: print(f"=== {title} ===")
    for name, v in m.items():
        print(f"  {name:15s}  mean={v['mean']:.4f}  median={v['median']:.4f}")
    print()

In [24]:
# Пересчитываем baseline, чтобы не зависеть от памяти (1 сек)
results_idx, scores = retriever.retrieve(query_tokens, k=100)
baseline_ranked = {
    qid: [product_ids[j] for j, s in zip(results_idx[i], scores[i]) if s > 0]
    for i, qid in enumerate(query_ids)
}

print_metrics(evaluate_retrieval(baseline_ranked), "Baseline BM25")

# Сегментация по числу Exact в разметке запроса
exact_counts = {qid: sum(1 for v in label_map[qid].values() if v == 2) for qid in query_ids}
buckets = {"0": [], "1-2": [], "3-9": [], "10+": []}
for qid in query_ids:
    n = exact_counts[qid]
    nd = ndcg_at_k_pessimistic(baseline_ranked[qid], qid)
    key = "0" if n == 0 else "1-2" if n <= 2 else "3-9" if n <= 9 else "10+"
    buckets[key].append(nd)

print("=== Сегментация по числу Exact в разметке ===")
for k, vals in buckets.items():
    print(f"  {k:>4s} Exact: n={len(vals):3d}  mean={np.mean(vals):.3f}  median={np.median(vals):.3f}")

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

=== Baseline BM25 ===
  NDCG@1_pess      mean=0.7177  median=1.0000
  NDCG@1_pool      mean=0.8125  median=1.0000
  NDCG@5_pess      mean=0.7150  median=0.8304
  NDCG@5_pool      mean=0.8110  median=1.0000
  NDCG@10_pess     mean=0.7127  median=0.7952
  NDCG@10_pool     mean=0.8150  median=0.9149
  MRR_exact        mean=0.5690  median=1.0000

=== Сегментация по числу Exact в разметке ===
     0 Exact: n=101  mean=0.616  median=0.685
   1-2 Exact: n= 68  mean=0.736  median=0.824
   3-9 Exact: n= 69  mean=0.634  median=0.652
   10+ Exact: n=242  mean=0.769  median=0.869


In [25]:
scored_low = sorted(
    [(qid, ndcg_at_k_pessimistic(baseline_ranked[qid], qid)) for qid in query_ids],
    key=lambda x: x[1],
)[:15]

print(f"{'qid':>4} {'ndcg':>6} {'exact':>6}  query")
for qid, sc in scored_low:
    qt = queries.loc[queries["query_id"] == qid, "query"].iloc[0]
    print(f"{qid:>4} {sc:>6.3f} {exact_counts[qid]:>6}  {qt!r}")

 qid   ndcg  exact  query
  84  0.000      1  'full metal bed rose gold'
  88  0.000      1  'rose gold lounge'
 125  0.000      1  'boho bed frame'
 136  0.000      0  'black fluffy stool'
 174  0.000      0  'star wars rug'
 179  0.000      0  'promo codes or discounts'
 207  0.000      0  'dull bed with shirt head board'
 222  0.000     31  'small loving roomtables'
 263  0.000     75  'small curtain rods'
 304  0.000      0  'merlyn 6'
 344  0.000      0  'ottoman bed queen'
 348  0.000      0  'one alium way'
 366  0.000      0  'drudge report'
 370  0.000      1  'rug for teen room'
 383  0.000      0  'minnestrista'


In [26]:
# Токенизируем названия один раз (~2 сек для 43k)
name_tokens_list = [tokenize(n) for n in products["product_name"]]
pid_to_name_tokens = dict(zip(products["product_id"], name_tokens_list))

def phrase_match(q_tokens, doc_tokens):
    """True, если q_tokens идут подряд в doc_tokens."""
    m = len(q_tokens)
    if m == 0 or m > len(doc_tokens):
        return False
    for i in range(len(doc_tokens) - m + 1):
        if doc_tokens[i:i+m] == q_tokens:
            return True
    return False

def apply_phrase_boost(bonus):
    """Переранжирует top-100 с фразовым бонусом."""
    out = {}
    for i, qid in enumerate(query_ids):
        q_tokens = query_tokens[i]
        rescored = []
        for pid, score in zip(
            [product_ids[j] for j, s in zip(results_idx[i], scores[i]) if s > 0],
            [s for s in scores[i] if s > 0],
        ):
            b = bonus if phrase_match(q_tokens, pid_to_name_tokens[pid]) else 0.0
            rescored.append((pid, float(score) + b))
        rescored.sort(key=lambda x: -x[1])
        out[qid] = [pid for pid, _ in rescored]
    return out

In [27]:
print_metrics(evaluate_retrieval(baseline_ranked), "Baseline (без boost)")

for bonus in [2, 5, 10, 20]:
    boosted = apply_phrase_boost(bonus)
    print_metrics(evaluate_retrieval(boosted), f"Phrase boost = {bonus}")

=== Baseline (без boost) ===
  NDCG@1_pess      mean=0.7177  median=1.0000
  NDCG@1_pool      mean=0.8125  median=1.0000
  NDCG@5_pess      mean=0.7150  median=0.8304
  NDCG@5_pool      mean=0.8110  median=1.0000
  NDCG@10_pess     mean=0.7127  median=0.7952
  NDCG@10_pool     mean=0.8150  median=0.9149
  MRR_exact        mean=0.5690  median=1.0000

=== Phrase boost = 2 ===
  NDCG@1_pess      mean=0.7208  median=1.0000
  NDCG@1_pool      mean=0.8156  median=1.0000
  NDCG@5_pess      mean=0.7165  median=0.8304
  NDCG@5_pool      mean=0.8115  median=1.0000
  NDCG@10_pess     mean=0.7146  median=0.7903
  NDCG@10_pool     mean=0.8157  median=0.9182
  MRR_exact        mean=0.5743  median=1.0000

=== Phrase boost = 5 ===
  NDCG@1_pess      mean=0.7208  median=1.0000
  NDCG@1_pool      mean=0.8156  median=1.0000
  NDCG@5_pess      mean=0.7165  median=0.8304
  NDCG@5_pool      mean=0.8115  median=1.0000
  NDCG@10_pess     mean=0.7145  median=0.7903
  NDCG@10_pool     mean=0.8157  median=0.9182

In [28]:
best_bonus = 5  # заменить на лучший по метрикам
boosted_best = apply_phrase_boost(best_bonus)

demo_qid = 1
print(f"Query: {queries.loc[queries['query_id'] == demo_qid, 'query'].iloc[0]!r}")
print(f"После phrase boost = {best_bonus}:\n")
for j, pid in enumerate(boosted_best[demo_qid][:10], 1):
    lab = label_map[demo_qid].get(pid)
    lab_str = {2: "Exact", 1: "Partial", 0: "Irrelevant", None: "UNJUDGED"}[lab]
    print(f"  {j:2d}. [{lab_str:10s}] {pid_to_name[pid][:70]}")

Query: 'smart coffee table'
После phrase boost = 5:

   1. [Exact     ] smart coffee table with storage
   2. [Partial   ] shaker coffee table
   3. [Partial   ] sanibel 3 legs coffee table
   4. [Partial   ] crotzer coffee table
   5. [Partial   ] sobro smart end table with built-in outlets
   6. [Partial   ] vandergriff wooden coffee table
   7. [UNJUDGED  ] stainless steel 3 piece coffee , tea , & sugar set
   8. [UNJUDGED  ] laguna 5 piece teak sofa seating group with cushions
   9. [UNJUDGED  ] bondi 4 piece sofa seating group set
  10. [UNJUDGED  ] litedge wi-fi accessible smart power strip socket plate


In [29]:
pid_to_name = dict(zip(products["product_id"], products["product_name"]))

def inspect_query(qid, top=15):
    qt = queries.loc[queries["query_id"] == qid, "query"].iloc[0]
    print(f"query_id={qid} | {qt!r}")
    print(f"Exact: {sum(1 for v in label_map[qid].values() if v==2)} | "
          f"Partial: {sum(1 for v in label_map[qid].values() if v==1)} | "
          f"Irrelevant: {sum(1 for v in label_map[qid].values() if v==0)}")
    print(f"NDCG@10 pess = {ndcg_at_k_pessimistic(baseline_ranked[qid], qid):.3f}\n")
    print(f"Топ-{top}:")
    for j, pid in enumerate(baseline_ranked[qid][:top], 1):
        lab = label_map[qid].get(pid)
        lab_str = {2:"Exact", 1:"Partial", 0:"Irrelevant", None:"UNJUDGED"}[lab]
        print(f"  {j:2d}. [{lab_str:10s}] {pid_to_name[pid][:70]}")
    print()

inspect_query(263)  # small curtain rods
inspect_query(222)  # small loving roomtables (opечатка)
inspect_query(344)  # ottoman bed queen

query_id=263 | 'small curtain rods'
Exact: 75 | Partial: 50 | Irrelevant: 166
NDCG@10 pess = 0.000

Топ-15:
   1. [UNJUDGED  ] adjustable tension curtain tension rod
   2. [UNJUDGED  ] premium button shower curtain hooks
   3. [UNJUDGED  ] premium roller shower curtain hooks
   4. [UNJUDGED  ] premium yin shower curtain hooks
   5. [UNJUDGED  ] premium crystal ball shower curtain hooks
   6. [UNJUDGED  ] wagner shower hooks
   7. [UNJUDGED  ] premium ora shower curtain hooks
   8. [UNJUDGED  ] premium globe shower curtain hooks
   9. [UNJUDGED  ] premium vitalia resin shower curtain hooks
  10. [UNJUDGED  ] premium time 12 piece shower curtain hook set
  11. [UNJUDGED  ] benge adjustable 0.44 '' tension rod
  12. [UNJUDGED  ] eire 72 '' adjustable straight tension shower curtain rod
  13. [UNJUDGED  ] tavernier decorative spring tension single curtain rod
  14. [Exact     ] alleman adjustable 0.625 '' single curtain rod
  15. [Exact     ] anferney single curtain rod

query_id=222 | 'sm

In [30]:
# Сколько товаров в каталоге вообще упоминают "curtain rod" в названии?
mask = products["product_name"].str.lower().str.contains("curtain rod", na=False)
print(f"Товаров с 'curtain rod' в названии: {mask.sum()}")

# Сколько из них размечены как Exact для qid=263?
exact_263 = [pid for pid, g in label_map[263].items() if g == 2]
print(f"Exact для qid=263: {len(exact_263)}")
print(f"Из них с 'curtain rod' в названии: "
      f"{sum(1 for pid in exact_263 if 'curtain rod' in pid_to_name[pid].lower())}")

Товаров с 'curtain rod' в названии: 119
Exact для qid=263: 75
Из них с 'curtain rod' в названии: 68


In [31]:
# Что вообще BM25 находит для этого запроса — распределение скоров
qt = tokenize("small curtain rods")
print(f"Query tokens: {qt}")
res, sc = retriever.retrieve([qt], k=50)
for j, (idx, s) in enumerate(zip(res[0][:20], sc[0][:20]), 1):
    pid = product_ids[idx]
    lab = label_map[263].get(pid)
    lab_str = {2:"Exact", 1:"Partial", 0:"Irrelevant", None:"UNJUDGED"}[lab]
    print(f"  {j:2d}. [{lab_str:10s}] {pid_to_name[pid][:60]:60s} (score={s:.2f})")

Query tokens: ['small', 'curtain', 'rod']


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

   1. [UNJUDGED  ] adjustable tension curtain tension rod                       (score=7.41)
   2. [UNJUDGED  ] premium button shower curtain hooks                          (score=7.29)
   3. [UNJUDGED  ] premium roller shower curtain hooks                          (score=7.27)
   4. [UNJUDGED  ] premium yin shower curtain hooks                             (score=7.24)
   5. [UNJUDGED  ] premium crystal ball shower curtain hooks                    (score=7.24)
   6. [UNJUDGED  ] wagner shower hooks                                          (score=7.23)
   7. [UNJUDGED  ] premium ora shower curtain hooks                             (score=7.23)
   8. [UNJUDGED  ] premium globe shower curtain hooks                           (score=7.23)
   9. [UNJUDGED  ] premium vitalia resin shower curtain hooks                   (score=7.22)
  10. [UNJUDGED  ] premium time 12 piece shower curtain hook set                (score=7.18)
  11. [UNJUDGED  ] benge adjustable 0.44 '' tension rod               

In [32]:
# Готовим тексты по полям (features — только ключи+значения, без описания)
import time

def features_text(raw):
    return " ".join(f"{k} {v}" for k, v in parse_features(raw))

t0 = time.time()
name_tokens_docs = [tokenize(n) for n in products["product_name"]]
feat_tokens_docs = [tokenize(features_text(f)) for f in tqdm(products["product_features"], desc="feat tokens")]
desc_tokens_docs = [tokenize(d) for d in products["product_description"]]
print(f"Токенизация полей: {time.time() - t0:.1f}s")

r_name = bm25s.BM25(); r_name.index(name_tokens_docs)
r_feat = bm25s.BM25(); r_feat.index(feat_tokens_docs)
r_desc = bm25s.BM25(); r_desc.index(desc_tokens_docs)

feat tokens: 100%|██████████| 42994/42994 [01:34<00:00, 452.88it/s]


Токенизация полей: 140.0s


BM25S Create Vocab:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/42994 [00:00<?, ?it/s]

...

BM25S Compute Scores:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Create Vocab:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/42994 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/42994 [00:00<?, ?it/s]

In [33]:
def _norm_scores(res_idx, scores, i):
    """Возвращает {pid: normalized_score} для запроса i. Нормализуем по max."""
    row = scores[i]
    mx = row.max() if row.max() > 0 else 1.0
    return {product_ids[j]: float(s) / mx for j, s in zip(res_idx[i], row) if s > 0}

def field_weighted_search(w_name, w_feat, w_desc, k_candidates=200, k_out=100):
    t0 = time.time()
    _, sn = r_name.retrieve(query_tokens, k=k_candidates)
    _, sf = r_feat.retrieve(query_tokens, k=k_candidates)
    _, sd = r_desc.retrieve(query_tokens, k=k_candidates)
    rn = r_name.retrieve(query_tokens, k=k_candidates)[0]
    rf = r_feat.retrieve(query_tokens, k=k_candidates)[0]
    rd = r_desc.retrieve(query_tokens, k=k_candidates)[0]

    out = {}
    for i, qid in enumerate(query_ids):
        n = _norm_scores(rn, sn, i)
        f = _norm_scores(rf, sf, i)
        d = _norm_scores(rd, sd, i)
        all_pids = set(n) | set(f) | set(d)
        scored = {
            pid: w_name * n.get(pid, 0) + w_feat * f.get(pid, 0) + w_desc * d.get(pid, 0)
            for pid in all_pids
        }
        ranked = sorted(scored.items(), key=lambda x: -x[1])[:k_out]
        out[qid] = [pid for pid, _ in ranked]
    print(f"field-weighted search: {time.time() - t0:.1f}s")
    return out

In [34]:
def field_weighted_search(w_name, w_feat, w_desc, k_candidates=200, k_out=100):
    t0 = time.time()
    rn, sn = r_name.retrieve(query_tokens, k=k_candidates)
    rf, sf = r_feat.retrieve(query_tokens, k=k_candidates)
    rd, sd = r_desc.retrieve(query_tokens, k=k_candidates)
    out = {}
    for i, qid in enumerate(query_ids):
        n = _norm_scores(rn, sn, i)
        f = _norm_scores(rf, sf, i)
        d = _norm_scores(rd, sd, i)
        all_pids = set(n) | set(f) | set(d)
        scored = {pid: w_name*n.get(pid,0) + w_feat*f.get(pid,0) + w_desc*d.get(pid,0)
                  for pid in all_pids}
        out[qid] = [pid for pid, _ in sorted(scored.items(), key=lambda x: -x[1])[:k_out]]
    print(f"field-weighted search: {time.time() - t0:.1f}s")
    return out

In [35]:
configs = [
    (1.0, 1.0, 1.0),   # all-equal
    (2.0, 1.0, 1.0),   # name×2
    (5.0, 1.0, 1.0),   # name×5
    (10.0, 1.0, 1.0),  # name×10
    (5.0, 1.0, 0.5),   # name×5, desc×0.5
    (5.0, 2.0, 1.0),   # name×5, feat×2
    (3.0, 1.5, 0.5),   # более сбалансированный
]

best = None
for w_n, w_f, w_d in configs:
    ranked = field_weighted_search(w_n, w_f, w_d)
    m = evaluate_retrieval(ranked)
    ndcg10 = m["NDCG@10_pess"]["mean"]
    ndcg1 = m["NDCG@1_pess"]["mean"]
    mrr = m["MRR_exact"]["mean"]
    marker = ""
    if best is None or ndcg10 > best[0]:
        best = (ndcg10, (w_n, w_f, w_d), ranked)
        marker = " *"
    print(f"w_name={w_n:4.1f} w_feat={w_f:4.1f} w_desc={w_d:4.1f} | "
          f"NDCG@1={ndcg1:.4f} NDCG@10={ndcg10:.4f} MRR={mrr:.4f}{marker}")

print(f"\nЛучший: {best[1]} с NDCG@10={best[0]:.4f}")

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

field-weighted search: 3.5s
w_name= 1.0 w_feat= 1.0 w_desc= 1.0 | NDCG@1=0.6708 NDCG@10=0.6777 MRR=0.4688 *


BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

...

field-weighted search: 3.4s
w_name= 5.0 w_feat= 2.0 w_desc= 1.0 | NDCG@1=0.7229 NDCG@10=0.7156 MRR=0.5286


BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

field-weighted search: 3.0s
w_name= 3.0 w_feat= 1.5 w_desc= 0.5 | NDCG@1=0.7083 NDCG@10=0.7062 MRR=0.5147

Лучший: (5.0, 1.0, 1.0) с NDCG@10=0.7213


In [36]:
def inspect_field_weighted(qid, w_name=5.0, w_feat=1.0, w_desc=1.0, top=15):
    """Смотрим field-weighted выдачу для конкретного запроса."""
    qt = queries.loc[queries["query_id"] == qid, "query"].iloc[0]
    print(f"query_id={qid} | {qt!r}\n")
    
    # Retrиваем по каждому полю отдельно — видно, откуда приходит сигнал
    rn, sn = r_name.retrieve([tokenize(qt)], k=50)
    rf, sf = r_feat.retrieve([tokenize(qt)], k=50)
    rd, sd = r_desc.retrieve([tokenize(qt)], k=50)
    
    # Показываем топ-5 из каждого поля
    for name, res, sc in [("NAME", rn, sn), ("FEAT", rf, sf), ("DESC", rd, sd)]:
        print(f"--- {name} top-5 ---")
        for idx, s in zip(res[0][:5], sc[0][:5]):
            pid = product_ids[idx]
            print(f"  {s:.2f}  {pid_to_name[pid][:65]}")
        print()
    
    # Итоговый field-weighted top
    fw_ranked = field_weighted_search(w_name, w_feat, w_desc, k_out=top)
    print(f"=== Field-weighted ({w_name},{w_feat},{w_desc}) top-{top} ===")
    for j, pid in enumerate(fw_ranked[qid], 1):
        lab = label_map[qid].get(pid)
        lab_str = {2:"Exact", 1:"Partial", 0:"Irrelevant", None:"UNJUDGED"}[lab]
        print(f"  {j:2d}. [{lab_str:10s}] {pid_to_name[pid][:65]}")
    print(f"\nNDCG@10 pess = {ndcg_at_k_pessimistic(fw_ranked[qid], qid):.3f}")

inspect_field_weighted(263)

query_id=263 | 'small curtain rods'



BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

--- NAME top-5 ---
  4.42  double curtain rod
  4.42  curtain single rod
  4.42  single curtain rod
  4.05  marciniak single curtain rod
  4.05  matias single curtain rod

--- FEAT top-5 ---
  5.71  lipson round urn adjustable single curtain rod
  5.28  payan single curtain rod
  5.25  adjustable tension curtain tension rod
  5.19  marc floral sheer rod pocket single curtain panel
  5.01  coulombe flat button drapery window double curtain rod

--- DESC top-5 ---
  6.66  premium button shower curtain hooks
  6.66  premium globe shower curtain hooks
  6.66  premium yin shower curtain hooks
  6.66  premium roller shower curtain hooks
  6.66  premium ora shower curtain hooks



BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

field-weighted search: 4.0s
=== Field-weighted (5.0,1.0,1.0) top-15 ===
   1. [Exact     ] anferney single curtain rod
   2. [Exact     ] mcgarvey single curtain rod
   3. [Exact     ] buderus single curtain rod
   4. [Exact     ] lysia single curtain rod
   5. [Exact     ] haubstadt single curtain rod
   6. [Partial   ] fatih single curtain rod
   7. [Exact     ] fade single curtain rod
   8. [Exact     ] deboer single curtain rod
   9. [Partial   ] deangelis single curtain rod
  10. [Exact     ] desjardins single curtain rod
  11. [Exact     ] deans single curtain rod
  12. [Exact     ] montreuil single curtain rod
  13. [Partial   ] katharina double curtain rod
  14. [Exact     ] silliman single curtain rod
  15. [Exact     ] vrinda single curtain rod

NDCG@10 pess = 0.928


In [37]:
# То же для опечатки и pool-bias кейса — для полноты картины
inspect_field_weighted(222)
inspect_field_weighted(344)

query_id=222 | 'small loving roomtables'



BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

--- NAME top-5 ---
  4.06  love is
  4.06  vintage love
  3.68  hands of love
  3.68  lovely peacock figurine
  3.68  masque of love

--- FEAT top-5 ---
  4.65  24k gold plated crystal studded 10 piece rose sculpture
  3.71  spoonful by olivia rose - textual art print on canvas
  3.67  full bloom xiii floral by danhui nai - graphic art print
  3.63  precious moments love at first sight elephant 4x6 resin and glass
  3.61  precious moments love at first sight elephant 4x6 resin and glass

--- DESC top-5 ---
  3.88  live love laugh zen garden
  3.09  oboyle meditation statue
  3.00  convertible sectional sofa couch , l-shaped couch with modern lin
  2.91  bolen 30.31 '' wide convertible chair
  2.89  clarissa twin 32 '' wide pillow back futon chair



...

--- NAME top-5 ---
  3.97  stromsburg queen platform bed
  3.97  masonville queen platform bed
  3.97  francheska queen standard bed
  3.97  queen tufted standard bed
  3.97  fabienne queen standard bed

--- FEAT top-5 ---
  4.82  giancarlo queen bed frame
  4.62  roxana metal bed frame
  4.18  kaela 46 '' metal heavy duty bed frame
  4.14  yoplac 38 '' standard profile bed frame
  4.14  frazier bed frame

--- DESC top-5 ---
  5.58  dipika tufted upholstered standard bed
  5.58  dipson tufted upholstered standard bed
  5.58  dippolito tufted upholstered standard bed
  5.35  elzadie queen four poster bed
  5.34  vela heavy duty bed frame



BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

field-weighted search: 2.7s
=== Field-weighted (5.0,1.0,1.0) top-15 ===
   1. [UNJUDGED  ] giancarlo queen bed frame
   2. [UNJUDGED  ] masonville queen platform bed
   3. [UNJUDGED  ] fabienne queen standard bed
   4. [UNJUDGED  ] elzadie queen four poster bed
   5. [UNJUDGED  ] sinead queen upholstered platform bed
   6. [UNJUDGED  ] carrollton queen upholstered standard bed
   7. [UNJUDGED  ] lamberton queen upholstered platform bed
   8. [UNJUDGED  ] karmakar queen upholstered platform bed
   9. [UNJUDGED  ] louque queen upholstered platform bed
  10. [UNJUDGED  ] ameed queen four poster bed
  11. [Partial   ] minick queen tufted standard bed
  12. [Irrelevant] mckeel queen upholstered standard bed
  13. [UNJUDGED  ] emilia queen tufted standard bed
  14. [Irrelevant] amed queen four poster bed
  15. [UNJUDGED  ] abbot diamond queen upholstered standard bed

NDCG@10 pess = 0.000


In [38]:
# Классифицируем «безнадёжные» запросы эвристикой:
# (a) в пуле 0 Exact и 0 Partial ≥ 3 — нечего искать
# (b) в пуле 1-2 Exact, но запрос содержит очевидную опечатку/не-товарное слово
hopeless_ids = set()

for qid in query_ids:
    exact_n = sum(1 for v in label_map[qid].values() if v == 2)
    partial_n = sum(1 for v in label_map[qid].values() if v == 1)
    if exact_n == 0 and partial_n < 5:
        hopeless_ids.add(qid)

# Добавим запросы с очевидной опечаткой (ручной список из худших)
manual_typos = {222, 348, 366, 383, 304, 179, 207, 136, 174, 344}  # сюда можно добавлять
hopeless_ids |= manual_typos

viable_ids = [q for q in query_ids if q not in hopeless_ids]
print(f"Viable: {len(viable_ids)}/{len(query_ids)} | Hopeless: {len(hopeless_ids)}")

# Считаем NDCG на двух срезах: все vs viable
best_ranked = field_weighted_search(5.0, 1.0, 1.0)
all_ndcg = np.mean([ndcg_at_k_pessimistic(best_ranked[q], q) for q in query_ids])
viable_ndcg = np.mean([ndcg_at_k_pessimistic(best_ranked[q], q) for q in viable_ids])
print(f"NDCG@10 на всех:    {all_ndcg:.4f}")
print(f"NDCG@10 на viable:  {viable_ndcg:.4f}")

Viable: 468/480 | Hopeless: 12


BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

field-weighted search: 2.6s
NDCG@10 на всех:    0.7213
NDCG@10 на viable:  0.7372


In [39]:
from sentence_transformers import SentenceTransformer
import time

# Билдим «короткий» документ: name + features, без длинного описания.
# Длинное описание размывает семантику названия.
def build_semantic_doc(row):
    feats = features_text(row["product_features"])[:400]  # обрезаем, чтобы не доминировало
    return f"{row['product_name']}. {feats}"

semantic_docs = products.apply(build_semantic_doc, axis=1).tolist()

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

t0 = time.time()
product_embs = model.encode(
    semantic_docs, batch_size=256, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
print(f"Product encoding: {time.time() - t0:.1f}s | shape={product_embs.shape}")

t0 = time.time()
query_embs = model.encode(
    query_texts, batch_size=256, show_progress_bar=False,
    normalize_embeddings=True, convert_to_numpy=True,
)
print(f"Query encoding: {time.time() - t0:.1f}s | shape={query_embs.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/168 [00:00<?, ?it/s]

Product encoding: 377.8s | shape=(42994, 384)
Query encoding: 0.7s | shape=(480, 384)


In [40]:
import faiss

# Inner product на нормализованных векторах = cosine similarity
index = faiss.IndexFlatIP(product_embs.shape[1])
index.add(product_embs)

t0 = time.time()
dense_scores, dense_idx = index.search(query_embs, 100)
print(f"Dense retrieval: {time.time() - t0:.1f}s")

dense_ranked = {
    qid: [product_ids[j] for j, s in zip(dense_idx[i], dense_scores[i]) if s > 0]
    for i, qid in enumerate(query_ids)
}

print_metrics(evaluate_retrieval(dense_ranked), "Pure dense (MiniLM-L6)")

Dense retrieval: 0.2s
=== Pure dense (MiniLM-L6) ===
  NDCG@1_pess      mean=0.6990  median=1.0000
  NDCG@1_pool      mean=0.7896  median=1.0000
  NDCG@5_pess      mean=0.6935  median=0.7770
  NDCG@5_pool      mean=0.7941  median=0.9065
  NDCG@10_pess     mean=0.6866  median=0.7671
  NDCG@10_pool     mean=0.7971  median=0.8783
  MRR_exact        mean=0.5028  median=0.5000



In [41]:
def rrf_fuse(ranked_a, ranked_b, k_rrf=60):
    """RRF: score(d) = Σ 1/(k + rank_i(d)). Устойчив к разным шкалам скоров."""
    fused = {}
    for qid in query_ids:
        s = {}
        for rank, pid in enumerate(ranked_a[qid], 1):
            s[pid] = s.get(pid, 0) + 1.0 / (k_rrf + rank)
        for rank, pid in enumerate(ranked_b[qid], 1):
            s[pid] = s.get(pid, 0) + 1.0 / (k_rrf + rank)
        fused[qid] = [pid for pid, _ in sorted(s.items(), key=lambda x: -x[1])]
    return fused

# Лучший BM25 конфиг (5,1,1) из 3g
fw_ranked = field_weighted_search(5.0, 1.0, 1.0)

for k_rrf in [10, 30, 60, 100]:
    fused = rrf_fuse(fw_ranked, dense_ranked, k_rrf=k_rrf)
    m = evaluate_retrieval(fused)
    print(f"RRF k={k_rrf:3d} | "
          f"NDCG@1={m['NDCG@1_pess']['mean']:.4f} "
          f"NDCG@10={m['NDCG@10_pess']['mean']:.4f} "
          f"MRR={m['MRR_exact']['mean']:.4f}")

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/480 [00:00<?, ?it/s]

field-weighted search: 4.4s
RRF k= 10 | NDCG@1=0.7760 NDCG@10=0.7413 MRR=0.5561
RRF k= 30 | NDCG@1=0.7875 NDCG@10=0.7596 MRR=0.5514
RRF k= 60 | NDCG@1=0.7958 NDCG@10=0.7649 MRR=0.5536
RRF k=100 | NDCG@1=0.7958 NDCG@10=0.7644 MRR=0.5531


In [42]:
from sentence_transformers import CrossEncoder
import time

# MiniLM-L6 — быстрый, ~200 пар/сек на CPU
# Если хватит терпения — попробуй позже L12 или electra-base, они на 2-3 пункта сильнее
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
print("Cross-encoder loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder loaded.


In [43]:
def build_rerank_text(row, max_feat_chars=300):
    """Короткий саммари документа: name + топ-фичи + класс. Без длинного description."""
    parts = [row["product_name"]]
    feats = features_text(row["product_features"])
    if feats:
        parts.append(feats[:max_feat_chars])
    if isinstance(row["product_class"], str):
        parts.append(row["product_class"])
    return " . ".join(p for p in parts if p)

products["rerank_text"] = products.progress_apply(build_rerank_text, axis=1)
pid_to_rerank_text = dict(zip(products["product_id"], products["rerank_text"]))

# Санити-чек
print(pid_to_rerank_text[products["product_id"].iloc[0]][:300])

100%|██████████| 42994/42994 [00:06<00:00, 6960.49it/s] 

solid wood platform bed . overallwidth-sidetoside 64.7 dsprimaryproductstyle modern countryoforigin-additionaldetails made in viet nam headboardtype slat headboardheight-toptobedframe 25.35 weightcapacity 500 estimatedtimetosetup 120 overalllength-headtotoe 77.9 levelofassembly full assembly needed 


In [44]:
def rerank_hybrid(rrf_ranked, k_candidates=100, batch_size=64):
    """
    Переранжирует top-k_candidates от RRF через cross-encoder.
    Возвращает {qid: [pid, ...]} с полной длиной top-k_candidates.
    """
    out = {}
    t0 = time.time()
    for i, qid in enumerate(tqdm(query_ids, desc="rerank")):
        cands = rrf_ranked[qid][:k_candidates]
        if not cands:
            out[qid] = []
            continue
        qtext = query_texts[i]
        pairs = [(qtext, pid_to_rerank_text[pid]) for pid in cands]
        scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        order = sorted(zip(cands, scores), key=lambda x: -x[1])
        out[qid] = [pid for pid, _ in order]
    print(f"rerank time: {time.time() - t0:.1f}s")
    return out

In [45]:
hybrid_ranked = rrf_fuse(fw_ranked, dense_ranked, k_rrf=60)

for k_cand in [50, 100]:
    print(f"\n--- rerank top-{k_cand} ---")
    reranked = rerank_hybrid(hybrid_ranked, k_candidates=k_cand)
    print_metrics(evaluate_retrieval(reranked), f"RRF + CrossEncoder ({k_cand})")


--- rerank top-50 ---


rerank: 100%|██████████| 480/480 [02:38<00:00,  3.03it/s]


rerank time: 158.4s
=== RRF + CrossEncoder (50) ===
  NDCG@1_pess      mean=0.7958  median=1.0000
  NDCG@1_pool      mean=0.8365  median=1.0000
  NDCG@5_pess      mean=0.7798  median=0.9127
  NDCG@5_pool      mean=0.8278  median=1.0000
  NDCG@10_pess     mean=0.7746  median=0.8817
  NDCG@10_pool     mean=0.8246  median=0.9306
  MRR_exact        mean=0.5843  median=1.0000


--- rerank top-100 ---


rerank: 100%|██████████| 480/480 [05:27<00:00,  1.46it/s]


rerank time: 327.9s
=== RRF + CrossEncoder (100) ===
  NDCG@1_pess      mean=0.7948  median=1.0000
  NDCG@1_pool      mean=0.8396  median=1.0000
  NDCG@5_pess      mean=0.7758  median=0.8920
  NDCG@5_pool      mean=0.8295  median=1.0000
  NDCG@10_pess     mean=0.7713  median=0.8870
  NDCG@10_pool     mean=0.8285  median=0.9337
  MRR_exact        mean=0.5850  median=1.0000



In [46]:
reranked_100 = rerank_hybrid(hybrid_ranked, k_candidates=100)

# Те же три «больных» запроса
for qid in [263, 222, 344]:
    qt = queries.loc[queries["query_id"] == qid, "query"].iloc[0]
    print(f"\n=== qid={qid} | {qt!r} ===")
    print(f"  NDCG@10: {ndcg_at_k_pessimistic(reranked_100[qid], qid):.3f}")
    for j, pid in enumerate(reranked_100[qid][:10], 1):
        lab = label_map[qid].get(pid)
        lab_str = {2:"Exact", 1:"Partial", 0:"Irrelevant", None:"UNJUDGED"}[lab]
        print(f"    {j:2d}. [{lab_str:10s}] {pid_to_name[pid][:65]}")

rerank: 100%|██████████| 480/480 [05:21<00:00,  1.49it/s]

rerank time: 321.8s

=== qid=263 | 'small curtain rods' ===
  NDCG@10: 0.710
     1. [UNJUDGED  ] adjustable tension curtain tension rod
     2. [Partial   ] berggren single curtain rod
     3. [Exact     ] lysia single curtain rod
     4. [Exact     ] haubstadt single curtain rod
     5. [Exact     ] montreuil single curtain rod
     6. [Exact     ] datus single curtain rod
     7. [Exact     ] alleman adjustable 0.625 '' single curtain rod
     8. [Exact     ] menelaus single curtain rod
     9. [Exact     ] gottlieb single curtain rod
    10. [Exact     ] gottschalk single curtain rod

=== qid=222 | 'small loving roomtables' ===
  NDCG@10: 0.263
     1. [Exact     ] vikki c table nesting tables
     2. [UNJUDGED  ] ellert 3 piece nesting tables
     3. [UNJUDGED  ] 4-piece counter height dining room table set , bar table with one
     4. [UNJUDGED  ] 5 piece dining bar table set counter height with 4 pub stool , mo
     5. [Partial   ] stella 3 piece dining table set , wooden kitche

In [47]:
# Финальный прогон всех методов и сводка
final = {
    "BM25 baseline (name×2+feat+desc+class)": evaluate_retrieval(baseline_ranked),
    "Field-weighted BM25 (5,1,1)":            evaluate_retrieval(fw_ranked),
    "Pure dense (MiniLM-L6)":                 evaluate_retrieval(dense_ranked),
    "RRF k=60 (hybrid)":                      evaluate_retrieval(hybrid_ranked),
    "RRF + CrossEncoder rerank":              evaluate_retrieval(reranked_100),
}

print(f"{'Метод':45s} {'NDCG@1':>8s} {'NDCG@5':>8s} {'NDCG@10':>8s} {'MRR':>8s}")
print("-" * 85)
for name, m in final.items():
    print(f"{name:45s} {m['NDCG@1_pess']['mean']:8.4f} {m['NDCG@5_pess']['mean']:8.4f} "
          f"{m['NDCG@10_pess']['mean']:8.4f} {m['MRR_exact']['mean']:8.4f}")

Метод                                           NDCG@1   NDCG@5  NDCG@10      MRR
-------------------------------------------------------------------------------------
BM25 baseline (name×2+feat+desc+class)          0.7177   0.7150   0.7127   0.5690
Field-weighted BM25 (5,1,1)                     0.7354   0.7255   0.7213   0.5358
Pure dense (MiniLM-L6)                          0.6990   0.6935   0.6866   0.5028
RRF k=60 (hybrid)                               0.7958   0.7685   0.7649   0.5536
RRF + CrossEncoder rerank                       0.7948   0.7758   0.7713   0.5850


In [48]:
def rerank_with_scores(rrf_ranked, k_candidates=50, batch_size=64):
    """Возвращает {qid: [(pid, ce_score), ...]}, отсортированный по ce_score."""
    out = {}
    for i, qid in enumerate(tqdm(query_ids, desc="rerank")):
        cands = rrf_ranked[qid][:k_candidates]
        if not cands:
            out[qid] = []
            continue
        pairs = [(query_texts[i], pid_to_rerank_text[pid]) for pid in cands]
        scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        out[qid] = sorted(zip(cands, scores.tolist()), key=lambda x: -x[1])
    return out

ce_scored = rerank_with_scores(hybrid_ranked, k_candidates=50)

rerank: 100%|██████████| 480/480 [02:42<00:00,  2.95it/s]


In [49]:
def minmax_norm(scores):
    lo, hi = min(scores), max(scores)
    if hi - lo < 1e-9:
        return [0.5] * len(scores)
    return [(s - lo) / (hi - lo) for s in scores]

def blend_rrf_ce(rrf_ranked, ce_scored, alpha=0.5, k_out=100):
    """
    alpha: вес CE (1-alpha — вес RRF на обратном ранге).
    Нормализуем оба сигнала в [0,1], смешиваем, сортируем.
    """
    out = {}
    for qid in query_ids:
        rrf_list = rrf_ranked[qid][:k_out]
        # RRF → [0,1] через обратный ранг
        rrf_score_map = {pid: 1.0 / (1 + rank) for rank, pid in enumerate(rrf_list)}
        if rrf_score_map:
            vals = list(rrf_score_map.values())
            lo, hi = min(vals), max(vals)
            if hi - lo > 1e-9:
                rrf_score_map = {p: (s - lo) / (hi - lo) for p, s in rrf_score_map.items()}

        # CE scores в [0,1]
        ce_map = {pid: s for pid, s in ce_scored.get(qid, [])}
        if ce_map:
            ce_vals = minmax_norm(list(ce_map.values()))
            ce_map = {pid: s for pid, s in zip(ce_map.keys(), ce_vals)}

        all_pids = set(rrf_score_map) | set(ce_map)
        blended = {
            pid: (1 - alpha) * rrf_score_map.get(pid, 0.0) + alpha * ce_map.get(pid, 0.0)
            for pid in all_pids
        }
        out[qid] = [pid for pid, _ in sorted(blended.items(), key=lambda x: -x[1])]
    return out

In [50]:
for alpha in [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]:
    blended = blend_rrf_ce(hybrid_ranked, ce_scored, alpha=alpha)
    m = evaluate_retrieval(blended)
    print(f"alpha={alpha:.1f} | "
          f"NDCG@1={m['NDCG@1_pess']['mean']:.4f} "
          f"NDCG@5={m['NDCG@5_pess']['mean']:.4f} "
          f"NDCG@10={m['NDCG@10_pess']['mean']:.4f} "
          f"MRR={m['MRR_exact']['mean']:.4f}")

alpha=0.0 | NDCG@1=0.7958 NDCG@5=0.7685 NDCG@10=0.7649 MRR=0.5535
alpha=0.3 | NDCG@1=0.7958 NDCG@5=0.7811 NDCG@10=0.7786 MRR=0.5612
alpha=0.5 | NDCG@1=0.7990 NDCG@5=0.7847 NDCG@10=0.7792 MRR=0.5697
alpha=0.7 | NDCG@1=0.8052 NDCG@5=0.7879 NDCG@10=0.7811 MRR=0.5809
alpha=0.9 | NDCG@1=0.7969 NDCG@5=0.7857 NDCG@10=0.7778 MRR=0.5838
alpha=1.0 | NDCG@1=0.7958 NDCG@5=0.7798 NDCG@10=0.7746 MRR=0.5846


In [52]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

e5_model = SentenceTransformer("intfloat/e5-small-v2")

# E5 требует префиксы. Документы — "passage: ", запросы — "query: "
# Используем те же semantic_docs, что и для MiniLM (name + короткие features)
e5_docs = ["passage: " + d for d in semantic_docs]
e5_queries = ["query: " + q for q in query_texts]

t0 = time.time()
e5_product_embs = e5_model.encode(
    e5_docs, batch_size=256, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
print(f"e5 product encoding: {time.time() - t0:.1f}s | shape={e5_product_embs.shape}")

t0 = time.time()
e5_query_embs = e5_model.encode(
    e5_queries, batch_size=256, show_progress_bar=False,
    normalize_embeddings=True, convert_to_numpy=True,
)
print(f"e5 query encoding: {time.time() - t0:.1f}s")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\ASUS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--intfloat--e5-small-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/168 [00:00<?, ?it/s]

e5 product encoding: 749.1s | shape=(42994, 384)
e5 query encoding: 1.1s


In [53]:
import faiss

e5_index = faiss.IndexFlatIP(e5_product_embs.shape[1])
e5_index.add(e5_product_embs)

dense_scores_e5, dense_idx_e5 = e5_index.search(e5_query_embs, 100)
e5_dense_ranked = {
    qid: [product_ids[j] for j, s in zip(dense_idx_e5[i], dense_scores_e5[i]) if s > 0]
    for i, qid in enumerate(query_ids)
}

print_metrics(evaluate_retrieval(e5_dense_ranked), "Pure dense (e5-small-v2)")

=== Pure dense (e5-small-v2) ===
  NDCG@1_pess      mean=0.7823  median=1.0000
  NDCG@1_pool      mean=0.8313  median=1.0000
  NDCG@5_pess      mean=0.7531  median=0.8539
  NDCG@5_pool      mean=0.8230  median=0.9529
  NDCG@10_pess     mean=0.7423  median=0.8245
  NDCG@10_pool     mean=0.8239  median=0.9055
  MRR_exact        mean=0.5643  median=1.0000



In [54]:
from sentence_transformers import CrossEncoder

# L12 — та же архитектура, 12 слоёв вместо 6
reranker_l12 = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", max_length=512)
print("L12 loaded.")

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

c:\Users\ASUS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

L12 loaded.


In [55]:
def rerank_with_scores(reranker, rrf_ranked, k_candidates=50, batch_size=64):
    """Возвращает {qid: [(pid, ce_score), ...]}, отсортированный по ce_score."""
    out = {}
    for i, qid in enumerate(tqdm(query_ids, desc="rerank")):
        cands = rrf_ranked[qid][:k_candidates]
        if not cands:
            out[qid] = []
            continue
        pairs = [(query_texts[i], pid_to_rerank_text[pid]) for pid in cands]
        scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        out[qid] = sorted(zip(cands, scores.tolist()), key=lambda x: -x[1])
    return out

# Гибрид: field-weighted BM25 + e5 dense
hybrid_e5 = rrf_fuse(fw_ranked, e5_dense_ranked, k_rrf=60)

# Переранжируем L12
ce_scored_l12_e5 = rerank_with_scores(reranker_l12, hybrid_e5, k_candidates=50)

rerank: 100%|██████████| 480/480 [06:41<00:00,  1.19it/s]


In [56]:
def blend_rrf_ce(rrf_ranked, ce_scored, alpha=0.5, k_out=100):
    out = {}
    for qid in query_ids:
        rrf_list = rrf_ranked[qid][:k_out]
        rrf_map = {pid: 1.0/(1+r) for r, pid in enumerate(rrf_list)}
        if rrf_map:
            vals = list(rrf_map.values()); lo, hi = min(vals), max(vals)
            if hi - lo > 1e-9:
                rrf_map = {p:(v-lo)/(hi-lo) for p, v in rrf_map.items()}
        ce_map = {pid: s for pid, s in ce_scored.get(qid, [])}
        if ce_map:
            ce_vals = minmax_norm(list(ce_map.values()))
            ce_map = {pid: s for pid, s in zip(ce_map.keys(), ce_vals)}
        all_pids = set(rrf_map) | set(ce_map)
        blended = {p: (1-alpha)*rrf_map.get(p,0) + alpha*ce_map.get(p,0) for p in all_pids}
        out[qid] = [pid for pid, _ in sorted(blended.items(), key=lambda x: -x[1])]
    return out

# Сетка alpha для прогона A
print("=== Прогон A: e5 + L12 ===")
for alpha in [0.3, 0.5, 0.7, 0.9, 1.0]:
    blended = blend_rrf_ce(hybrid_e5, ce_scored_l12_e5, alpha=alpha)
    m = evaluate_retrieval(blended)
    print(f"alpha={alpha:.1f} | NDCG@1={m['NDCG@1_pess']['mean']:.4f} "
          f"NDCG@5={m['NDCG@5_pess']['mean']:.4f} "
          f"NDCG@10={m['NDCG@10_pess']['mean']:.4f} "
          f"MRR={m['MRR_exact']['mean']:.4f}")

=== Прогон A: e5 + L12 ===
alpha=0.3 | NDCG@1=0.7781 NDCG@5=0.7788 NDCG@10=0.7785 MRR=0.5749
alpha=0.5 | NDCG@1=0.7833 NDCG@5=0.7840 NDCG@10=0.7797 MRR=0.5795
alpha=0.7 | NDCG@1=0.8010 NDCG@5=0.7855 NDCG@10=0.7801 MRR=0.5893
alpha=0.9 | NDCG@1=0.8104 NDCG@5=0.7855 NDCG@10=0.7795 MRR=0.5925
alpha=1.0 | NDCG@1=0.7927 NDCG@5=0.7816 NDCG@10=0.7775 MRR=0.5776


In [57]:
reranker_electra = CrossEncoder("cross-encoder/ms-marco-electra-base", max_length=512)

# Важно: Electra в 3 раза медленнее L12. 480×50 пар займёт ~15–20 минут
ce_scored_electra_e5 = rerank_with_scores(
    reranker_electra, hybrid_e5, k_candidates=50, batch_size=32
)

print("=== Прогон B: e5 + Electra-base ===")
for alpha in [0.3, 0.5, 0.7, 0.9, 1.0]:
    blended = blend_rrf_ce(hybrid_e5, ce_scored_electra_e5, alpha=alpha)
    m = evaluate_retrieval(blended)
    print(f"alpha={alpha:.1f} | NDCG@1={m['NDCG@1_pess']['mean']:.4f} "
          f"NDCG@5={m['NDCG@5_pess']['mean']:.4f} "
          f"NDCG@10={m['NDCG@10_pess']['mean']:.4f} "
          f"MRR={m['MRR_exact']['mean']:.4f}")

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

c:\Users\ASUS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--cross-encoder--ms-marco-electra-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

rerank: 100%|██████████| 480/480 [21:32<00:00,  2.69s/it]


=== Прогон B: e5 + Electra-base ===
alpha=0.3 | NDCG@1=0.7781 NDCG@5=0.7759 NDCG@10=0.7721 MRR=0.5756
alpha=0.5 | NDCG@1=0.7917 NDCG@5=0.7779 NDCG@10=0.7747 MRR=0.5814
alpha=0.7 | NDCG@1=0.7917 NDCG@5=0.7820 NDCG@10=0.7747 MRR=0.5771
alpha=0.9 | NDCG@1=0.7979 NDCG@5=0.7821 NDCG@10=0.7754 MRR=0.5875
alpha=1.0 | NDCG@1=0.7854 NDCG@5=0.7774 NDCG@10=0.7739 MRR=0.5727


In [58]:
# Гибрид: field-weighted BM25 + MiniLM-L6 dense (как было)
hybrid_l6 = rrf_fuse(fw_ranked, dense_ranked, k_rrf=60)

# Переранжируем L12
ce_scored_l12_l6 = rerank_with_scores(reranker_l12, hybrid_l6, k_candidates=50)

print("=== Прогон C: MiniLM-L6 dense + L12 CE ===")
for alpha in [0.3, 0.5, 0.7, 0.9, 1.0]:
    blended = blend_rrf_ce(hybrid_l6, ce_scored_l12_l6, alpha=alpha)
    m = evaluate_retrieval(blended)
    print(f"alpha={alpha:.1f} | NDCG@1={m['NDCG@1_pess']['mean']:.4f} "
          f"NDCG@5={m['NDCG@5_pess']['mean']:.4f} "
          f"NDCG@10={m['NDCG@10_pess']['mean']:.4f} "
          f"MRR={m['MRR_exact']['mean']:.4f}")

rerank: 100%|██████████| 480/480 [07:39<00:00,  1.04it/s]


=== Прогон C: MiniLM-L6 dense + L12 CE ===
alpha=0.3 | NDCG@1=0.7958 NDCG@5=0.7817 NDCG@10=0.7783 MRR=0.5601
alpha=0.5 | NDCG@1=0.7990 NDCG@5=0.7855 NDCG@10=0.7791 MRR=0.5679
alpha=0.7 | NDCG@1=0.8010 NDCG@5=0.7858 NDCG@10=0.7821 MRR=0.5777
alpha=0.9 | NDCG@1=0.8042 NDCG@5=0.7877 NDCG@10=0.7801 MRR=0.5883
alpha=1.0 | NDCG@1=0.8000 NDCG@5=0.7816 NDCG@10=0.7769 MRR=0.5846


In [59]:
# После всех прогонов — сохраняем всё, что пригодится
import pickle
np.save("data/e5_product_embs.npy", e5_product_embs)
with open("data/ce_scored_l12_e5.pkl", "wb") as f: pickle.dump(ce_scored_l12_e5, f)
with open("data/ce_scored_l12_l6.pkl", "wb") as f: pickle.dump(ce_scored_l12_l6, f)
with open("data/ce_scored_electra_e5.pkl", "wb") as f: pickle.dump(ce_scored_electra_e5, f)
with open("data/hybrid_e5.pkl", "wb") as f: pickle.dump(hybrid_e5, f)
print("Все артефакты сохранены.")

Все артефакты сохранены.


In [60]:
def rrf_fuse_n(ranked_lists, k_rrf=60, weights=None):
    """RRF над произвольным числом ранжирований. weights — опциональные веса."""
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    out = {}
    for qid in query_ids:
        s = {}
        for ranked, w in zip(ranked_lists, weights):
            for rank, pid in enumerate(ranked[qid], 1):
                s[pid] = s.get(pid, 0) + w / (k_rrf + rank)
        out[qid] = [pid for pid, _ in sorted(s.items(), key=lambda x: -x[1])]
    return out

# 3-way: fw BM25 + MiniLM dense + e5 dense
hybrid_3way = rrf_fuse_n([fw_ranked, dense_ranked, e5_dense_ranked], k_rrf=60)
print_metrics(evaluate_retrieval(hybrid_3way), "3-way RRF (BM25 + MiniLM + e5)")

# С L12 CE
ce_scored_3way = rerank_with_scores(reranker_l12, hybrid_3way, k_candidates=50)

print("\n=== 3-way + L12 CE ===")
for alpha in [0.3, 0.5, 0.7, 0.9, 1.0]:
    blended = blend_rrf_ce(hybrid_3way, ce_scored_3way, alpha=alpha)
    m = evaluate_retrieval(blended)
    print(f"alpha={alpha:.1f} | NDCG@1={m['NDCG@1_pess']['mean']:.4f} "
          f"NDCG@5={m['NDCG@5_pess']['mean']:.4f} "
          f"NDCG@10={m['NDCG@10_pess']['mean']:.4f} "
          f"MRR={m['MRR_exact']['mean']:.4f}")

=== 3-way RRF (BM25 + MiniLM + e5) ===
  NDCG@1_pess      mean=0.7927  median=1.0000
  NDCG@1_pool      mean=0.8260  median=1.0000
  NDCG@5_pess      mean=0.7766  median=0.8920
  NDCG@5_pool      mean=0.8232  median=0.9475
  NDCG@10_pess     mean=0.7705  median=0.8592
  NDCG@10_pool     mean=0.8235  median=0.9240
  MRR_exact        mean=0.5612  median=1.0000



rerank: 100%|██████████| 480/480 [06:35<00:00,  1.21it/s]



=== 3-way + L12 CE ===
alpha=0.3 | NDCG@1=0.7927 NDCG@5=0.7817 NDCG@10=0.7812 MRR=0.5662
alpha=0.5 | NDCG@1=0.7958 NDCG@5=0.7852 NDCG@10=0.7830 MRR=0.5711
alpha=0.7 | NDCG@1=0.7937 NDCG@5=0.7853 NDCG@10=0.7818 MRR=0.5756
alpha=0.9 | NDCG@1=0.8073 NDCG@5=0.7855 NDCG@10=0.7816 MRR=0.5858
alpha=1.0 | NDCG@1=0.7948 NDCG@5=0.7814 NDCG@10=0.7787 MRR=0.5809


In [61]:
# Пересечение top-10 разных методов для каждого запроса
def jaccard_at_k(ranked_a, ranked_b, qid, k=10):
    a = set(ranked_a[qid][:k])
    b = set(ranked_b[qid][:k])
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

pairs = [
    ("BM25 vs MiniLM", fw_ranked, dense_ranked),
    ("BM25 vs e5",     fw_ranked, e5_dense_ranked),
    ("MiniLM vs e5",   dense_ranked, e5_dense_ranked),
]
print("Средний Jaccard@10 между методами (ниже = более разные):")
for name, a, b in pairs:
    vals = [jaccard_at_k(a, b, qid) for qid in query_ids]
    print(f"  {name:20s}  {np.mean(vals):.3f}")

Средний Jaccard@10 между методами (ниже = более разные):
  BM25 vs MiniLM        0.131
  BM25 vs e5            0.188
  MiniLM vs e5          0.197


In [62]:
def bootstrap_ndcg(ranked, n_boot=1000, k=10, seed=42):
    rng = np.random.default_rng(seed)
    scores = np.array([ndcg_at_k_pessimistic(ranked[qid], qid, k) for qid in query_ids])
    n = len(scores)
    means = [scores[rng.integers(0, n, n)].mean() for _ in range(n_boot)]
    return scores.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

configs_to_compare = {
    "Baseline (MiniLM+L6, a=0.7)": blend_rrf_ce(hybrid_l6, ce_scored, alpha=0.7),
    "Run C (MiniLM+L12, a=0.7)":   blend_rrf_ce(hybrid_l6, ce_scored_l12_l6, alpha=0.7),
    "Run A (e5+L12, a=0.7)":       blend_rrf_ce(hybrid_e5, ce_scored_l12_e5, alpha=0.7),
}

print(f"{'Config':35s} {'NDCG@10':>10s}  {'95% CI':>25s}")
for name, ranked in configs_to_compare.items():
    mean, lo, hi = bootstrap_ndcg(ranked)
    print(f"{name:35s} {mean:>10.4f}  [{lo:.4f}, {hi:.4f}]")

Config                                 NDCG@10                     95% CI
Baseline (MiniLM+L6, a=0.7)             0.7811  [0.7572, 0.8049]
Run C (MiniLM+L12, a=0.7)               0.7821  [0.7581, 0.8056]
Run A (e5+L12, a=0.7)                   0.7801  [0.7568, 0.8020]


In [63]:
import pickle
import numpy as np

# Финальный конфиг: Run C
FINAL_CONFIG = {
    "name": "MiniLM-L6 dense + L12 CE rerank",
    "dense_model": "sentence-transformers/all-MiniLM-L6-v2",
    "reranker":    "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "alpha":       0.7,
    "k_rrf":       60,
    "k_candidates": 50,
    "metrics": {
        "NDCG@1":  0.8010,
        "NDCG@5":  0.7858,
        "NDCG@10": 0.7821,
        "MRR":     0.5777,
    }
}

# Сохраняем финальный CE-скоринг — он самый дорогой в пересчёте
with open("data/final_ce_scored.pkl", "wb") as f:
    pickle.dump(ce_scored_l12_l6, f)

# Промежуточные результаты тоже сохраняем — вдруг пригодятся
with open("data/ce_scored_l6_l6.pkl", "wb") as f:       # baseline L6
    pickle.dump(ce_scored, f)
with open("data/hybrid_minilm.pkl", "wb") as f:         # hybrid для Run C
    pickle.dump(hybrid_l6, f)

# Метаданные финала
import json
with open("data/final_config.json", "w") as f:
    json.dump(FINAL_CONFIG, f, indent=2)

print("Финальные артефакты сохранены:")
print("  data/final_ce_scored.pkl  — CE-скоры Run C")
print("  data/hybrid_minilm.pkl    — RRF-выдача Run C")
print("  data/product_embs.npy     — эмбеддинги MiniLM (из шага 4)")
print("  data/final_config.json    — метаданные конфига")

Финальные артефакты сохранены:
  data/final_ce_scored.pkl  — CE-скоры Run C
  data/hybrid_minilm.pkl    — RRF-выдача Run C
  data/product_embs.npy     — эмбеддинги MiniLM (из шага 4)
  data/final_config.json    — метаданные конфига


In [64]:
final_summary = {
    "BM25 (name×2 + feat + desc + class)":     evaluate_retrieval(baseline_ranked),
    "Field-weighted BM25 (5,1,1)":             evaluate_retrieval(fw_ranked),
    "Pure dense (MiniLM-L6)":                  evaluate_retrieval(dense_ranked),
    "Pure dense (e5-small-v2)":                evaluate_retrieval(e5_dense_ranked),
    "RRF k=60 (BM25 + MiniLM)":                evaluate_retrieval(hybrid_l6),
    "RRF + L6 CE (baseline)":                  evaluate_retrieval(blend_rrf_ce(hybrid_l6, ce_scored, alpha=0.7)),
    "RRF + L12 CE (FINAL, alpha=0.7)":         evaluate_retrieval(blend_rrf_ce(hybrid_l6, ce_scored_l12_l6, alpha=0.7)),
    "3-way RRF + L12 CE (alpha=0.5)":          evaluate_retrieval(blend_rrf_ce(hybrid_3way, ce_scored_3way, alpha=0.5)),
}

print(f"{'Метод':45s} {'NDCG@1':>8s} {'NDCG@5':>8s} {'NDCG@10':>8s} {'MRR':>8s}")
print("─" * 85)
for name, m in final_summary.items():
    star = " ★" if "FINAL" in name else ""
    print(f"{name:45s} {m['NDCG@1_pess']['mean']:8.4f} {m['NDCG@5_pess']['mean']:8.4f} "
          f"{m['NDCG@10_pess']['mean']:8.4f} {m['MRR_exact']['mean']:8.4f}{star}")

Метод                                           NDCG@1   NDCG@5  NDCG@10      MRR
─────────────────────────────────────────────────────────────────────────────────────
BM25 (name×2 + feat + desc + class)             0.7177   0.7150   0.7127   0.5690
Field-weighted BM25 (5,1,1)                     0.7354   0.7255   0.7213   0.5358
Pure dense (MiniLM-L6)                          0.6990   0.6935   0.6866   0.5028
Pure dense (e5-small-v2)                        0.7823   0.7531   0.7423   0.5643
RRF k=60 (BM25 + MiniLM)                        0.7958   0.7685   0.7649   0.5536
RRF + L6 CE (baseline)                          0.8052   0.7879   0.7811   0.5809
RRF + L12 CE (FINAL, alpha=0.7)                 0.8010   0.7858   0.7821   0.5777 ★
3-way RRF + L12 CE (alpha=0.5)                  0.7958   0.7852   0.7830   0.5711
